# Demand Forecasting with Classical NLP and LLM Review Signals

Each product-month is a forecast origin: information available in month t forecasts review-count demand in t+1.

- Tier 0: Naive Baseline: Current-month as prediction for next.
- Tier 1: Classic Baseline: Current and historical demand lags + ratings/dispersion + target-month cyclical seasonality.
- Tier 2: Classical NLP: Tier 1 + current/lagged VADER sentiment + LDA topics.
- Tier 3: Advanced LLM Only: Tier 1 + 10 current/lagged LLM-extracted behavioral and semantic signals.
- Tier 4: Hybrid (NLP + LLM): Tier 2 + 10 current/lagged LLM-extracted behavioral and semantic signals.


### 0. Imports & Configuration
Set up variables and fixed configuration for any processing further down the line. + set random state so results are reproducable

Colab has some package clashes, need to uninstall and reinstall some packages (only needed if rerunning LLM)

In [ ]:
#!pip uninstall -y torch torchvision torchaudio vllm

In [ ]:
#!pip install torch vllm

In [ ]:
!pip install optuna #needed for hyperparameter optimization
!pip install gensim #needed for lda


In [ ]:
import os
import gc
import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
#import vllm commented out since llm signals are already extracted from reviews
#import torch
from optuna.samplers import TPESampler
from tqdm.auto import tqdm
from joblib import dump
from datetime import datetime

from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet, PoissonRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/LML_V2/V4/cohort_reviews_sampled_2013_2019V2.parquet"
CHECKPOINT_PATH = "/content/drive/MyDrive/LML_V2/V4/llm_extracted_checkpoint_2013_2019V2.parquet"
OUTPUT_POST_LLM_PATH = "/content/drive/MyDrive/LML_V2/V4/cohort_reviews_sampled_2013_2019.parquet"
LDA_CHECKPOINT_PATH = "/content/drive/MyDrive/Thesis Code/df_with_lda_topics.parquet"

LLM_SCHEMA_VERSION = "2013_2019"

DEBUG_SAMPLE_SIZE = None
TEST_FRACTION = 0.20
RANDOM_STATE = 123
N_LDA_TOPICS = 12
LDA_MAX_FEATURES = 1000
LLM_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
#LLM_MODEL_ID = "microsoft/Phi-4-mini-instruct"
LLM_BATCH_SIZE = 32
LLM_SAVE_EVERY = 2500
STORE_RAW_LLM_OUTPUT = True

LLM_MAX_NEW_TOKENS = 64
LLM_MAX_INPUT_TOKENS = 1500
LLM_MAX_REVIEW_TOKENS = 300

llm_cols = ['repurchase_intent_positive', 'repurchase_intent_negative', 'great_value_for_money', 'poor_value_for_money', 'product_performance_failure', 'product_change_or_quality_decline', 'sensory_or_usability_complaint', 'adverse_health_or_safety_effect', 'damaged_or_incorrect_when_received', 'social_or_trend_driven_purchase']

start_date = datetime(2013, 1, 1)
end_date = datetime(2019, 12, 31)

### 1. Check out the data
Ingests the raw reviews and do some schema validation. Ensures basic data quality (e.g., verifying datetimes, detecting missing prices) before any complex text processing begins.

In [ ]:
#check if data is in expected path
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"DATA_PATH does not exist:\n{DATA_PATH}\n"
    )

df = pd.read_parquet(DATA_PATH).reset_index(drop=True)

df = df[(df.datetime >= start_date) & (df.datetime <= end_date)]

#if in previous cell DEBUG_SAMPLE_SIZE is given, only run following cells with that specific sample
if DEBUG_SAMPLE_SIZE is not None:
    df = (df.sample(n=min(DEBUG_SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE).reset_index(drop=True).copy())

#added title to 'review_text_full' -> since all code already written taking 'review_text', just replace that one
df['review_text'] = df['review_text_full']
required_cols = [
    "product_id",
    "rating",
    "timestamp",
    "review_text",
    "price",
    "datetime",
    "year_month",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["review_text"] = df["review_text"].fillna("").astype(str)
df["year_month"] = df["datetime"].dt.to_period("M")

print(f"Rows:              {len(df):,}")
print(f"Columns:           {len(df.columns):,}")
print(f"Unique products:   {df['product_id'].nunique():,}")
print(f"First month:       {df['year_month'].min()}")
print(f"Last month:        {df['year_month'].max()}")
print(f"Rating missing:    {df['rating'].isna().sum():,}")
print(f"Review text empty: {(df['review_text'].str.len() == 0).sum():,}")

### 2. Split data into train/test (chronologically)
 Use strict temporal cutoff so to guarantee that future consumer behavior does not leak into any fitting of the models.

In [ ]:
all_months = np.array(sorted(df["year_month"].dropna().unique()))

split_idx = int(len(all_months) * (1 - TEST_FRACTION))
split_idx = min(max(split_idx, 1), len(all_months) - 1)
train_cutoff = all_months[split_idx]

train_reviews_mask = df["year_month"] < train_cutoff
test_reviews_mask = df["year_month"] >= train_cutoff

print(f"Chronological cutoff: {train_cutoff}")
print(f"Training review rows: {train_reviews_mask.sum():,}")
print(f"Holdout review rows:  {test_reviews_mask.sum():,}")
print(f"Training months:      {df.loc[train_reviews_mask, 'year_month'].nunique():,}")
print(f"Holdout months:       {df.loc[test_reviews_mask, 'year_month'].nunique():,}")

### 3. EDA

In [ ]:
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

df_eda = df.copy()
df_eda['Split'] = np.where(df_eda['year_month'] < train_cutoff, 'Training (t)', 'Holdout (t+1)')
df_eda['review_length'] = df_eda['review_text'].str.len()

eda_ts = df_eda.groupby('year_month').agg(
    review_count=('product_id', 'count'),
    avg_rating=('rating', 'mean'),
    avg_price=('price', 'mean')
).reset_index()

eda_ts['timestamp_plot'] = eda_ts['year_month'].dt.to_timestamp()
cutoff_timestamp = pd.Period(train_cutoff, freq='M').to_timestamp()

plt.figure(figsize=(14, 5))
sns.lineplot(data=eda_ts, x='timestamp_plot', y='review_count', marker="o", color="#1f77b4", linewidth=2)
plt.axvline(x=cutoff_timestamp, color='red', linestyle='--', linewidth=2, label=f'Holdout Cutoff ({train_cutoff})')
plt.title('Monthly Review Volume (Demand proxy) Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Total Reviews')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 5))
sns.lineplot(data=eda_ts, x='timestamp_plot', y='avg_rating', marker="s", color="#2ca02c", linewidth=2)
plt.axvline(x=cutoff_timestamp, color='red', linestyle='--', linewidth=2, label=f'Holdout Cutoff ({train_cutoff})')
plt.title('Average Monthly Product Rating Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Average Rating (1-5 Stars)')
plt.ylim(1, 5.2)
plt.legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(data=df_eda, x='rating', hue='Split', multiple='dodge', discrete=True, shrink=0.8, ax=axes[0], palette='Set2')
axes[0].set_title('Rating Distribution: Training vs. Holdout', fontweight='bold')
axes[0].set_xticks([1, 2, 3, 4, 5])
axes[0].set_xlabel('Rating')

sns.kdeplot(data=df_eda, x='review_length', hue='Split', log_scale=True, ax=axes[1], fill=True, palette='Set2', common_norm=False)
axes[1].set_title('Review Text Length Distribution (Log Scale)', fontweight='bold')
axes[1].set_xlabel('Number of Characters in Review')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
corr_cols = ['rating', 'price', 'review_length']
corr_matrix = df_eda[corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.3f', linewidths=0.5)
plt.title('Correlation Matrix of Base Features', fontweight='bold')
plt.tight_layout()
plt.show()

del df_eda, eda_ts
gc.collect()

### 4. Tier 2A: Sentiment Analysis
Calculates continuous score for how negative/positive a review is (-1 to 1)

In [ ]:
import nltk
nltk.download("vader_lexicon", quiet=True)

sia = SentimentIntensityAnalyzer()
df["sentiment_score"] = df["review_text"].map(
    lambda x: sia.polarity_scores(x)["compound"]
)

print("VADER sentiment:")
print(df["sentiment_score"].describe().round(3))

In [ ]:
#Before doing LLM extraction, do some checks
# Define unique identifier columns (used for restarted from checkpoint)
UNIQUE_ID_COLS = ["product_id", "user_id", "timestamp"]

# Find all rows that share the exact same combination of these columns
duplicate_mask = df.duplicated(subset=UNIQUE_ID_COLS, keep=False)
duplicates_df = df[duplicate_mask]

if duplicates_df.empty:
    print(f"Yay combi of {UNIQUE_ID_COLS} is unique")
    print(f"Total rows verified: {len(df):,}")
else:
    # Calculate how many distinct groups of duplicates exist
    num_duplicate_groups = duplicates_df.drop_duplicates(subset=UNIQUE_ID_COLS).shape[0]

    print(f"Nooope, found {len(duplicates_df):,} total overlapping rows across {num_duplicate_groups:,} unique combinations.")
    print("First few overlapping groups:")

    # Sort by the ID columns so the duplicates are grouped together for easy reading
    display(duplicates_df.sort_values(by=UNIQUE_ID_COLS)[UNIQUE_ID_COLS + ["review_text"]].head(20))

### 6. Tier 3: LLM Zero-Shot Extraction
Using a local 3-billion-parameter Instruction LLM (Qwen2.5), perform strict, deterministic (as possible) multi-label classification --> explicitly extract 10 behavioral and operational signals (e.g., functional product failure, arrival damage) while trying to keep it from halucinating

In [ ]:
import os
import gc
import time
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer

# function to check the output of the llm
def parse_llm_output(result, llm_cols):
    raw = str(result).strip()
    cleaned = raw.lower().strip()
    parsed = {col: False for col in llm_cols}
    unexpected_tags = []
    parse_ok = True
    # if none output, quit
    if cleaned == "none":
        return parsed, True, []
    categorys = [category.strip() for category in cleaned.replace("\n", ",").split(",") if category.strip()]
    # check for no labels in output, quit
    if not categorys:
        return parsed, False, ["EMPTY_OUTPUT"]
    # sometimes "none" is given as tag, record this case
    if "none" in categorys:
        parse_ok = False
        unexpected_tags.append("NONE_WITH_OTHER_TAGS")
        categorys = [t for t in categorys if t != "none"]
    # check which actual requested tags were present
    seen = set()
    for tag in categorys:
        if tag in llm_cols:
            if tag not in seen:
                parsed[tag] = True
                seen.add(tag)
        else:
            parse_ok = False
            unexpected_tags.append(tag)

    return parsed, parse_ok, unexpected_tags


# Define columns that create a guaranteed unique fingerprint for a review
UNIQUE_ID_COLS = ["product_id", "user_id", "timestamp"]
# Create a stable composite key to prevent positional merge errors
df["_safe_merge_id"] = df[UNIQUE_ID_COLS].astype(str).agg('-'.join, axis=1)
#above is used since extracting the llm features takes a while, and using this can be restarted from a checkpoint

# since it takes a while to run, save output every 2500 reviews, check if llm labels are present/not NA and only continue with those
if os.path.exists(CHECKPOINT_PATH):
    print("Loading existing LLM checkpoint:", CHECKPOINT_PATH)
    checkpoint = pd.read_parquet(CHECKPOINT_PATH)
    errors = []
    # following checks to make sure data that was loaded in this run is the same as was saved to checkpoint
    missing = [col for col in llm_cols if col not in checkpoint.columns]
    if missing:
        errors.append(f"missing required LLM columns: {missing}")
    if "llm_schema_version" not in checkpoint.columns:
        errors.append("missing llm_schema_version")
    elif not checkpoint["llm_schema_version"].dropna().eq(LLM_SCHEMA_VERSION).all():
        errors.append(f"schema version is not exclusively {LLM_SCHEMA_VERSION!r}")

    # Generate identical key for the checkpoint
    if not errors:
        checkpoint["_safe_merge_id"] = checkpoint[UNIQUE_ID_COLS].astype(str).agg('-'.join, axis=1)
        # Make sure key is unique, prevent many-to-many merge
        if df["_safe_merge_id"].duplicated().any():
            errors.append("The provided UNIQUE_ID_COLS do not uniquely identify each review. Cannot safely align data.")
    if errors:
        raise ValueError("Incompatible LLM checkpoint; old taxonomy cannot be reused: " + "; ".join(errors))

    # Restore columns by mapping on the unique ID
    cols_to_restore = [c for c in llm_cols + ["llm_schema_version", "raw_llm_output", "llm_parse_ok", "llm_unexpected_tags"] if c in checkpoint.columns]

    # Ensures the exact row receives the exact label, regardless of sorting
    checkpoint_indexed = checkpoint.set_index("_safe_merge_id")
    for col in cols_to_restore:
        df[col] = df["_safe_merge_id"].map(checkpoint_indexed[col])
else:
    print("No LLM checkpoint found; starting fresh.")

# cleanup temporary merge ID
df = df.drop(columns=["_safe_merge_id"], errors='ignore')

# check if llm columns are already added to df (so if it was loaded from checkpoint), if not, generate them, fill with NA
for col in llm_cols:
    if col not in df.columns:
        df[col] = pd.Series(pd.array([pd.NA] * len(df), dtype="boolean"), index=df.index)
for col, dtype in [("raw_llm_output", "string"), ("llm_parse_ok", "boolean"), ("llm_unexpected_tags", "string")]:
    if col not in df.columns:
        df[col] = pd.Series(pd.array([pd.NA] * len(df), dtype=dtype), index=df.index)

needs_llm = df[llm_cols].isna().any(axis=1)
pending_indices = df.index[needs_llm].tolist()
print(f"Rows requiring LLM extraction: {len(pending_indices):,} / {len(df):,}")

# Run llm for all reviews which have no llm tags yet
if pending_indices:
    # Import vLLM specific modules
    from vllm import LLM, SamplingParams

    # import pretrained tokenizer from huggingface
    tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)

    # Initialize vLLM engine
    print("Loading vLLM engine...")
    # Set environment variable before vLLM loads to prevent multiprocess crashing in Colab
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
    llm = LLM(
        model=LLM_MODEL_ID,
        dtype="half",
        gpu_memory_utilization=0.85,  # Lowered slightly to give Colab's OS room to breathe
        max_model_len=4096,
        enable_prefix_caching=True,
        enforce_eager=True,           # disables CUDA graphs which was crashing Colab
        disable_log_stats=True
    )

    sampling_params = SamplingParams(
        temperature=0.0, #dont want the llm to get creative
        max_tokens=LLM_MAX_NEW_TOKENS
    )

    SYSTEM_PROMPT = """
You are a strict multi-label classifier for e-commerce customer reviews.
Classify ONLY signals explicitly supported by the review text.
RULES:
- Do not infer unstated intent, causes, defects, health effects, or purchase behavior.
- Do not classify from general sentiment or star rating alone.
- Each label requires its own explicit evidence.
- Multi-label is allowed only when each label is independently supported.
- Output ONLY exact label names from the list below, comma-separated, in the listed order.
- If none apply, output exactly: none
1. repurchase_intent_positive
Explicit future or repeat PURCHASE behavior: buy again, reorder, repurchase, subscribe,
order another, or continue buying.
Examples:
"Will buy again."
"Ordering another bottle."
"I keep repurchasing it."
Do NOT use for:
love, recommendation, favorite product, continued use, or long-time use without
explicit purchase/reorder behavior.
2. repurchase_intent_negative
Explicit statement that the reviewer will NOT buy/order/repurchase again.
Examples:
"Never buying this again."
"Won't repurchase."
Do NOT use for generic dissatisfaction or "would not recommend."
3. great_value_for_money
The review must explicitly connect the product to PRICE, MONEY, COST, VALUE,
AFFORDABILITY, or what is received for the amount paid.
VALID:
"Great value for the price."
"Worth the money."
"Good quality for what it costs."
"Half the price and works just as well."
"Competitive price for this quality."
"You get a lot for the money."
INVALID:
"Great product."
"Excellent quality."
"Works really well."
"Best one I've tried."
"Cheap."
"Inexpensive."
"Good price."
A positive product opinion alone is NOT value-for-money.
If there is no explicit economic/value comparison, do NOT use this label.
4. poor_value_for_money
The review must explicitly connect a negative judgment to PRICE, MONEY, COST,
VALUE, AFFORDABILITY, or what is received for the amount paid.
VALID:
"Not worth the money."
"Too expensive for what you get."
"Overpriced for such a small amount."
"Poor value for the price."
INVALID:
"Bad product."
"Poor quality."
"Too small."
"Doesn't work."
"Expensive."
A negative product opinion or price mention alone is NOT poor value-for-money.
5. product_performance_failure
Product explicitly fails its intended function, is ineffective, stops working,
or fails during normal use.
Examples:
"Doesn't work."
"The pump stopped working."
"The mirror fogs up every time."
Do NOT use for arrival damage, bad smell, health reactions, or merely being too small.
6. product_change_or_quality_decline
Explicit comparison between THIS product now and an earlier version or previous
purchase showing change or decline.
Examples:
"They changed the formula."
"This used to be better."
"My previous bottle was better."
A past-versus-present comparison of the same product is required.
7. sensory_or_usability_complaint
Explicit complaint about smell, taste, texture, feel, comfort, fit, application,
handling, size-for-use, or ease of use.
Examples:
"Smells awful."
"Too sticky."
"Hard to apply."
"The drawers are too small to fit my items."
8. adverse_health_or_safety_effect
Using the product ACTUALLY caused a negative physical or health effect.
Examples:
"It gave me a rash."
"My scalp started burning."
"It caused breakouts."
Do NOT use for:
possible risk, ingredient criticism, fear, "not safe", or positive statements such
as "did not cause irritation."
The physical effect must actually have happened.
9. damaged_or_incorrect_when_received
Explicit order/receipt problem already present at delivery, receipt, or first opening.
Examples:
"Arrived broken."
"Received it leaking."
"The package was open when delivered."
"I ordered 3.4 oz but received 1.7 oz."
"An item was missing."
"The wrong product was sent."
If receipt, delivery, first opening, or an explicit order mismatch is NOT stated,
do NOT use this label.
Do NOT use for:
broke later, stopped working later, fragile, too small for the user's needs,
or ordinary product failure.
10. social_or_trend_driven_purchase
Social media, influencer, viral hype, or an online trend explicitly influenced
the purchase.
Examples:
"Bought this because I saw it on TikTok."
"An influencer made me try it."
"I wanted to see if the viral hype was real."
Do NOT use for:
friend recommendations, ordinary customer reviews, or merely saying something
is popular.
IMPORTANT:
- Recommendation is NOT repurchase intent.
- Continued use is NOT repurchase intent.
- Product failure is NOT an arrival problem.
- Arrival problems are NOT automatically performance failures.
- Possible health risk is NOT an actual adverse health effect.
- Generic poor quality is NOT product_change_or_quality_decline.
- Do not invent labels.
Return ONLY exact matching label names, comma-separated, in the fixed order above.
If none apply, return exactly:
none
"""

    pbar = tqdm(total=len(pending_indices), desc="vLLM Extraction", unit="review", dynamic_ncols=True) #show progress
    start_time = time.time()

    # Chunk the data into blocks of 2500 so checkpoints are saved periodically
    for chunk_start in range(0, len(pending_indices), LLM_SAVE_EVERY):
        chunk_indices = pending_indices[chunk_start : chunk_start + LLM_SAVE_EVERY]

        # Format the prompts
        formatted_prompts = []
        for idx in chunk_indices:
            truncated_review = str(df.at[idx, "review_text"])[:1200]
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Review: {truncated_review}"}
            ]
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            formatted_prompts.append(prompt)

        # Put entire chunk into LLM.
        outputs = llm.generate(formatted_prompts, sampling_params, use_tqdm=False)

        # Parse output using function from before
        batch_records = []
        for i, output in enumerate(outputs):
            idx = chunk_indices[i]
            generated_text = output.outputs[0].text
            parsed, parse_ok, unexpected = parse_llm_output(generated_text, llm_cols)
            record = {col: parsed[col] for col in llm_cols}
            record["llm_parse_ok"] = parse_ok
            record["llm_unexpected_tags"] = ",".join(unexpected)
            record["raw_llm_output"] = generated_text
            record["llm_schema_version"] = LLM_SCHEMA_VERSION

            batch_records.append(record)
            pbar.update(1)

        # 4. Save checkpoint after chunk completes
        update_df = pd.DataFrame(batch_records, index=chunk_indices)
        for col_name in update_df.columns:
            df.loc[chunk_indices, col_name] = update_df[col_name]

        df.to_parquet(CHECKPOINT_PATH, index=False)

    pbar.close()

    # Clean up memory
    del llm
    gc.collect()
    import torch
    torch.cuda.empty_cache()

    print(f"LLM extraction finished in {(time.time() - start_time) / 60:.1f} minutes.")
else:
    print("All LLM labels are already present, extraction skipped")

df["llm_schema_version"] = LLM_SCHEMA_VERSION
for col in llm_cols:
    df[col] = df[col].astype("boolean")
#df.to_parquet(OUTPUT_POST_LLM_PATH, index=False)
print("Saved review-level enriched data:", OUTPUT_POST_LLM_PATH)

### 5. Tier 2B: LDA Topic Modeling
Unsupervised extraction of 12 LDA topics (determined using coherence and perplexity). The model is fitted only on the training window to prevent lookahead bias, and are then applied also to the holdout set (so after training on train data)

In [ ]:
sys.path.append("/content/drive/MyDrive/Thesis Code")
from run_lda import run_lda

topic_cols = [f"topic_{i+1}_prob" for i in range(N_LDA_TOPICS)]
# check if previous result already exists
lda_ready = False
if os.path.exists(LDA_CHECKPOINT_PATH):
    saved_df = pd.read_parquet(LDA_CHECKPOINT_PATH)
    lda_ready = (
        len(saved_df) == len(df)
        and all(col in saved_df.columns for col in topic_cols)
        and not saved_df[topic_cols].isna().any().any()
    )

# load checkpoint or run if it doesnt exist
if lda_ready:
    print("Loading checkpoint...")
    df = saved_df
else:
    print("No LDA checkpoint found. Running LDA for all reviews...")
    # takes about 20-30 min
    train_text = df.loc[
        train_reviews_mask,
        "review_text"
    ].fillna("")
    all_text = df["review_text"].fillna("")
    lda_topics = run_lda(
        train_text=train_text,
        all_text=all_text,
        n_topics=N_LDA_TOPICS
    )
    df[topic_cols] = lda_topics
    df.to_parquet(LDA_CHECKPOINT_PATH,index=False)
    print("LDA complete and df saved.")
print(f"LDA ready: {len(df):,} rows, {N_LDA_TOPICS} topics.")

### 7. Review LLM output
Check LLM parsing success rate, label co-occurrence, and any contradictions.

In [ ]:
#display for each tag in what percentage of reviews they are present
signal_summary = pd.DataFrame({"hits": df[llm_cols].fillna(False).sum(), "pct_reviews": df[llm_cols].fillna(False).mean().mul(100)}).sort_values("pct_reviews", ascending=False)
display(signal_summary.round(3))
#what percentage of reviews were parsed successfully, and inspect which didnt
parse_success_rate = df["llm_parse_ok"].mean()
print(f"LLM parser success rate: {parse_success_rate:.2%}")
display(df.loc[~df["llm_parse_ok"].fillna(False), ["raw_llm_output", "llm_unexpected_tags"]].head(30))
#how many tags per review?
llm_numeric = df[llm_cols].fillna(False).astype(int)
df["n_llm_signals"] = llm_numeric.sum(axis=1)
signal_count_distribution = df["n_llm_signals"].value_counts(normalize=True).sort_index().mul(100).rename("pct_reviews")
display(signal_count_distribution)

#check correlation between llm tags
llm_corr = llm_numeric.corr()
display(llm_corr.round(2))
cooccurrence = llm_numeric.T.dot(llm_numeric)
display(cooccurrence)

#check contradictions
for left, right in [("repurchase_intent_positive", "repurchase_intent_negative"), ("great_value_for_money", "poor_value_for_money")]:
    conflict = df[left].fillna(False) & df[right].fillna(False)
    print(f"Contradiction {left} + {right}: {int(conflict.sum()):,}")
    display(df.loc[conflict, ["review_text", "rating", "raw_llm_output"]].head(10))
for col in llm_cols:
    examples = df.loc[df[col].fillna(False), ["review_text", "rating", "raw_llm_output"]]
    print(f"\nManual examples - {col} ({len(examples):,} hits)")
    display(examples.sample(n=min(10, len(examples)), random_state=RANDOM_STATE) if len(examples) else examples)

#inspect tags
if DEBUG_SAMPLE_SIZE is not None:
    damaged_performance_audit = df.loc[
        df["damaged_or_incorrect_when_received"].fillna(False)
        & df["product_performance_failure"].fillna(False),
        ["product_id", "rating", "review_text", "raw_llm_output"],
    ].copy()
    product_change_audit = df.loc[
        df["product_change_or_quality_decline"].fillna(False),
        ["product_id", "rating", "review_text", "raw_llm_output"] + llm_cols,
    ].copy()
    repurchase_positive_audit = df.loc[
        df["repurchase_intent_positive"].fillna(False),
        ["product_id", "rating", "review_text", "raw_llm_output"],
    ].copy()
    adverse_health_audit = df.loc[
        df["adverse_health_or_safety_effect"].fillna(False),
        ["product_id", "rating", "review_text", "raw_llm_output"],
    ].copy()
    print(f"Damaged/incorrect + performance audit rows: {len(damaged_performance_audit):,}")
    display(damaged_performance_audit)
    print(f"Product-change audit rows: {len(product_change_audit):,}")
    display(product_change_audit)
    print(f"Positive-repurchase audit rows: {len(repurchase_positive_audit):,}")
    display(repurchase_positive_audit)
    print(f"Adverse-health audit rows: {len(adverse_health_audit):,}")
    display(adverse_health_audit)

In [ ]:
df.columns

### 8. Product-Month Aggregation
Want to eventually do the forecasting on product-month level. Hence, need to aggregate all generated features and categories up to this level. E.g. take the mean of llm flag poor_value_for_money over the whole month for a certain product.

In [ ]:
#exclude malformed LLM responses from feature aggregation

valid_llm_parse = df["llm_parse_ok"].fillna(False)

print(    f"Excluding {(~valid_llm_parse).sum():,} malformed LLM responses "
    "from monthly LLM feature denominators.")
df.loc[~valid_llm_parse, llm_cols] = pd.NA

#create final df which will be used for model training and evaluation
print("[Aggregation] Building product-month panel...")
aggregation_dict = {
    "review_text": "count",
    "rating": "mean",
    "price": "mean",
    "sentiment_score": "mean",
}
for col in topic_cols:
    aggregation_dict[col] = "mean"
for col in llm_cols:
    aggregation_dict[col] = "mean"

ts_df = (
    df.groupby(["product_id", "year_month"])
      .agg(aggregation_dict)
      .reset_index()
      .rename(columns={"review_text": "demand_volume", "rating": "avg_rating"})
)

#also create feature for "spread" of rating
rating_dispersion = (
    df.groupby(["product_id", "year_month"])["rating"]
      .agg(lambda x: x.std(ddof=0) if len(x) > 1 else 0.0)
      .reset_index(name="rating_std")
)
ts_df = ts_df.merge(rating_dispersion, on=["product_id", "year_month"], how="left")
ts_df = ts_df.sort_values(["product_id", "year_month"]).reset_index(drop=True)

print(f"Review-level rows:        {len(df):,}")
print(f"Product-month rows:       {len(ts_df):,}")
print(f"Unique products:          {ts_df['product_id'].nunique():,}")
print(f"Number of LDA topic cols:  {len(topic_cols)}")
rating_std_nonzero = (ts_df["rating_std"] > 0).mean()
print(f"Product-months with non-zero rating dispersion: {rating_std_nonzero:.2%}")
print(ts_df["rating_std"].describe())
monthly_llm_summary = []
for col in llm_cols:
    nonzero = ts_df[col] > 0
    monthly_llm_summary.append({"feature": col, "nonzero_product_months": int(nonzero.sum()), "pct_product_months_nonzero": float(nonzero.mean() * 100), "mean_monthly_rate": float(ts_df[col].mean()), "median_monthly_rate": float(ts_df[col].median()), "max_monthly_rate": float(ts_df[col].max())})
monthly_llm_summary = pd.DataFrame(monthly_llm_summary)
display(monthly_llm_summary)
for row in monthly_llm_summary.itertuples(index=False):
    if row.pct_product_months_nonzero < 1:
        print(f"WARNING: {row.feature} occurs in only {row.pct_product_months_nonzero:.1f}% of product-months.")

### 9. Continuous Product Timelines & Gap Filling

In [ ]:
#create a continuous, gap-free monthly calendar for each product by filling in missing months with zeros for review activity and carrying forward the previous month's ratings
filled_dfs = []
for product, group in tqdm(ts_df.groupby("product_id"), total=ts_df["product_id"].nunique(), desc="Building product calendars"):
    group = group.set_index("year_month")
    full_calendar = pd.period_range(start=group.index.min(), end=group.index.max(), freq="M")
    group = group.reindex(full_calendar)
    group["product_id"] = product
    group["demand_volume"] = group["demand_volume"].fillna(0)

    group["sentiment_score"] = group["sentiment_score"].fillna(0)
    for col in topic_cols:
        group[col] = group[col].fillna(0)
    for col in llm_cols:
        group[col] = group[col].fillna(0)

    for col in ["avg_rating", "rating_std", "price"]:
        group[col] = group[col].ffill()
    filled_dfs.append(group.reset_index(names="year_month"))

ts_continuous = pd.concat(filled_dfs, ignore_index=True)
ts_continuous = ts_continuous.sort_values(["product_id", "year_month"]).reset_index(drop=True)
print(f"Continuous product-month rows: {len(ts_continuous):,}")

### 10. Feature Engineering
Computes historical lags (e.g., demand in t-1, t-2) and cyclical features (sin/cos of the month) fore seasonal awareness

In [ ]:
#generate lagged features
print("[Feature Engineering] Building leakage-safe features...")
ts_continuous["target_year_month"] = ts_continuous["year_month"] + 1
target_month_num = ts_continuous["target_year_month"].dt.month
ts_continuous["target_month_sin"] = np.sin(2 * np.pi * target_month_num / 12.0)
ts_continuous["target_month_cos"] = np.cos(2 * np.pi * target_month_num / 12.0)

lag1_cols = ["demand_volume", "avg_rating", "rating_std", "sentiment_score", "price"] + topic_cols + llm_cols
for col in lag1_cols:
    ts_continuous[f"{col}_lag_1"] = ts_continuous.groupby("product_id")[col].shift(1)

ts_continuous["demand_volume_lag_2"] = ts_continuous.groupby("product_id")["demand_volume"].shift(2)
ts_continuous["demand_volume_rolling_mean_3"] = ts_continuous.groupby("product_id")["demand_volume"].transform(lambda s: s.rolling(window=3, min_periods=1).mean())
ts_continuous["price_change_1m"] = ts_continuous["price"] - ts_continuous["price_lag_1"]
ts_continuous["target_next_month_demand"] = ts_continuous.groupby("product_id")["demand_volume"].shift(-1)

model_df = ts_continuous.copy()
print(f"Forecast-origin rows before modeling filter: {len(model_df):,}")

### 11. Define Feature Tiers
Prepare the different data teirs

In [ ]:
t1_feats = [
    "demand_volume",
    "demand_volume_lag_1",
    "demand_volume_lag_2",
    "demand_volume_rolling_mean_3",
    "avg_rating",
    "avg_rating_lag_1",
    "rating_std",
    "rating_std_lag_1",
    "target_month_sin",
    "target_month_cos",
]

t2_feats = (
    t1_feats
    + [
        "sentiment_score",
        "sentiment_score_lag_1",
    ]
    + topic_cols
    + [f"{c}_lag_1" for c in topic_cols]
)

t3_feats = t1_feats + llm_cols + [f"{c}_lag_1" for c in llm_cols]

t4_feats = t2_feats + llm_cols + [f"{c}_lag_1" for c in llm_cols]

target_col = "target_next_month_demand"

all_required_model_cols = sorted(set(t4_feats + [target_col]))
common_complete_mask = model_df[all_required_model_cols].notna().all(axis=1)
model_complete = model_df.loc[common_complete_mask].copy()

print("=" * 70); print("FEATURE TIERS"); print("=" * 70)
print(f"Tier 1 features:           {len(t1_feats)}")
print(f"Tier 2 features:           {len(t2_feats)}")
print(f"Tier 3 features:           {len(t3_feats)}")
print(f"Tier 4 features:           {len(t4_feats)}")
print(f"Common complete-case rows: {len(model_complete):,}")

### 12. Pre-Modeling Feature Validation
Last check to make sure the data that will be used for forecasting is all good

In [ ]:
print("Ten features:", llm_cols)
print("\nReview-level prevalence:")
display(signal_summary.round(3))
print("\nProduct-month prevalence:")
display(monthly_llm_summary.round(4))
print(f"\nParser success rate: {parse_success_rate:.2%}")
print(f"Malformed outputs: {int((~df['llm_parse_ok'].fillna(False)).sum()):,}")
print("\nTags per review:")
display(signal_count_distribution)
print("\nCo-occurrence:")
display(cooccurrence)
print("\nPairwise correlations:")
display(llm_corr.round(2))
for left, right in [("repurchase_intent_positive", "repurchase_intent_negative"), ("great_value_for_money", "poor_value_for_money")]:
    print(f"Contradiction {left} + {right}: {int((df[left].fillna(False) & df[right].fillna(False)).sum()):,}")

### 13. Chronological Train & Holdout Split
Use the previously defined train and test cutof to prepare respective training and test data

In [ ]:
eval_months = np.array(sorted(model_complete["year_month"].unique()))
train_cutoff_eval = train_cutoff
if train_cutoff_eval not in set(eval_months):
    raise ValueError(
        f"Chosen cutoff {train_cutoff_eval} is not present in the complete modeling panel.")
train_data = (
    model_complete.loc[model_complete["year_month"] < train_cutoff_eval]
    .sort_values(["year_month", "product_id"])
    .reset_index(drop=True))

test_data = (
    model_complete.loc[model_complete["year_month"] >= train_cutoff_eval]
    .sort_values(["year_month", "product_id"])
    .reset_index(drop=True))

y_train = train_data[target_col]
y_test = test_data[target_col]

print(f"Cutoff forecast month:       {train_cutoff_eval}")
print(f"Train rows:                  {len(train_data):,}")
print(f"Holdout rows:                {len(test_data):,}")
print(f"Train months:                {train_data['year_month'].nunique():,}")
print(f"Holdout months:              {test_data['year_month'].nunique():,}")
print(f"Target mean (train):         {y_train.mean():.4f}")
print(f"Target mean (holdout):       {y_test.mean():.4f}")

### 14. Prep Cross-Validation
Preparing the expanding window cross validation by splitting the data in train and validation sets across 3 cross validation folds

In [ ]:
#use expanding window cross validation
unique_train_months = np.array(sorted(train_data["year_month"].unique()))

VAL_MONTHS = 4
N_CV_FOLDS = 3 #so that in total validation is done over a whole year
MIN_TRAIN_MONTHS = train_data['year_month'].nunique() - N_CV_FOLDS * VAL_MONTHS


required_months = ( MIN_TRAIN_MONTHS + N_CV_FOLDS * VAL_MONTHS)

if len(unique_train_months) < required_months:
    raise ValueError(
        f"Need at least {required_months} training months for this CV design, "
        f"but only {len(unique_train_months)} are available.")
cv_splits = []
for fold in range(N_CV_FOLDS):
    train_end = (MIN_TRAIN_MONTHS + fold * VAL_MONTHS)
    val_start = train_end
    val_end = val_start + VAL_MONTHS

    fold_train_months = unique_train_months[:train_end]
    fold_val_months = unique_train_months[val_start:val_end]

    assert len(fold_train_months) == (MIN_TRAIN_MONTHS + fold * VAL_MONTHS)
    assert len(fold_val_months) == VAL_MONTHS
    assert set(fold_train_months).isdisjoint(set(fold_val_months))
    assert (fold_train_months.max() < fold_val_months.min())

    fold_train_idx = np.flatnonzero(train_data["year_month"].isin(fold_train_months).to_numpy())
    fold_val_idx = np.flatnonzero(train_data["year_month"].isin(fold_val_months).to_numpy())
    cv_splits.append((fold_train_idx,fold_val_idx,))

    print(
        f"Fold {fold + 1}: "
        f"train months {fold_train_months.min()} "
        f"to {fold_train_months.max()} "
        f"({len(fold_train_months)} months, "
        f"{len(fold_train_idx):,} rows); "
        f"validation months {fold_val_months.min()} "
        f"to {fold_val_months.max()} "
        f"({len(fold_val_months)} months, "
        f"{len(fold_val_idx):,} rows)"
    )

#check if any months unused
cv_end = (MIN_TRAIN_MONTHS + N_CV_FOLDS * VAL_MONTHS)
remaining_train_months = unique_train_months[cv_end:]
if len(remaining_train_months) > 0:
    print(
        "\nTraining months not used for CV validation "
        "but included in final refit: "
        f"{remaining_train_months.min()} "
        f"to {remaining_train_months.max()} "
        f"({len(remaining_train_months)} months)")

### Model training and hyperparameter tuning
Iterate through the 4-Tier feature data sets, and use Bayesian hyperparameter tuning from the Optuna package to find the optimal model configurations.

In [ ]:
print("training...")
feature_sets = {
    "Tier 1: Baseline": t1_feats,
    "Tier 2: Classical NLP": t2_feats,
    "Tier 3: Advanced LLM Only": t3_feats,
    "Tier 4: Hybrid (NLP + LLM)": t4_feats,
}

results, fitted_models, test_predictions, tuning_objects = [], {}, {}, {}
#Functions to evaluate runs:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def evaluate_predictions(y_true, y_pred):
    return {"RMSE": rmse(y_true, y_pred), "MAE": float(mean_absolute_error(y_true, y_pred)), "R2": float(r2_score(y_true, y_pred))}

def clipped_rmse_loss(y_true, y_pred):
    return rmse(np.asarray(y_true), np.clip(np.asarray(y_pred), 0, None))

#using clipped scorer to ensure robust evaluation, avoiding effects of extreme outliers
clipped_rmse_scorer = make_scorer(clipped_rmse_loss, greater_is_better=False)

#FUnctions to tune the different models
def tune_elastic_net(X, y):
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(max_iter=20000, random_state=RANDOM_STATE)),
    ])
    param_grid = {
        "model__alpha": [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0],
        "model__l1_ratio": [0.0, 0.10, 0.25, 0.50, 0.75, 0.90, 1.0],
    }
    search = GridSearchCV(
        estimator=pipeline, param_grid=param_grid, scoring=clipped_rmse_scorer,
        cv=cv_splits, refit=True, n_jobs=-1
    )
    search.fit(X, y)
    return search

def tune_poisson(X, y):
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", PoissonRegressor(max_iter=5000)),
    ])
    search = GridSearchCV(
        estimator=pipeline, param_grid={"model__alpha": [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]},
        scoring=clipped_rmse_scorer, cv=cv_splits, refit=True, n_jobs=-1
    )
    search.fit(X, y)
    return search

#Use optuna tuning random forest
def tune_rf_optuna(X, y, n_trials=30, n_estimators=500, n_estimators_search=150):
    y_array = np.asarray(y)
    folds = [(X.iloc[tr].to_numpy(), X.iloc[val].to_numpy(), y_array[tr], y_array[val])
             for tr, val in cv_splits]

    def objective(trial):
        params = {
            "n_estimators": n_estimators_search,
            "max_depth": trial.suggest_categorical("max_depth", [4, 6, 8, 10, 12, 16, None]),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "max_features": trial.suggest_float("max_features", 0.30, 1.00),
            "bootstrap": True,
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
        }
        fold_rmse = []
        for step, (X_tr, X_val, y_tr, y_val) in enumerate(folds):
            model = RandomForestRegressor(**params)
            model.fit(X_tr, y_tr)
            pred = np.clip(model.predict(X_val), 0, None)
            fold_rmse.append(rmse(y_val, pred))
            trial.report(float(np.mean(fold_rmse)), step=step)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(fold_rmse))

    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=RANDOM_STATE),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=1),
    )
    study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=True)

    best_params = {**study.best_params, "n_estimators": n_estimators,
                    "bootstrap": True, "random_state": RANDOM_STATE, "n_jobs": -1}
    best_model = RandomForestRegressor(**best_params).fit(X, y)
    return best_model, study, best_params, float(study.best_value)

#Tuning xgb:
def tune_xgb_optuna(X, y, n_trials=40, max_estimators=2000, early_stopping_rounds=50):
    y_array = np.asarray(y)
    folds = [(X.iloc[tr].to_numpy(), X.iloc[val].to_numpy(), y_array[tr], y_array[val])
             for tr, val in cv_splits]

    def objective(trial):
        params = {
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "n_estimators": max_estimators,
            "max_depth": trial.suggest_int("max_depth", 2, 6),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0, log=True),
            "subsample": trial.suggest_float("subsample", 0.60, 1.00),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.50, 1.00),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 5.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.10, 20.0, log=True),
            "tree_method": "hist",
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
            "verbosity": 0,
            "early_stopping_rounds": early_stopping_rounds,
        }
        fold_rmse = []
        for step, (X_tr, X_val, y_tr, y_val) in enumerate(folds):
            model = xgb.XGBRegressor(**params)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            pred = np.clip(model.predict(X_val), 0, None)
            fold_rmse.append(rmse(y_val, pred))
            trial.report(float(np.mean(fold_rmse)), step=step)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(fold_rmse))

    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=RANDOM_STATE),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=1),
    )
    study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=True)

    best_params = {**study.best_params, "objective": "reg:squarederror", "eval_metric": "rmse",
                    "tree_method": "hist", "random_state": RANDOM_STATE, "n_jobs": -1, "verbosity": 0}

    n_val = max(int(0.1 * len(X)), 500)
    X_fit, X_final_val = X.iloc[:-n_val].to_numpy(), X.iloc[-n_val:].to_numpy()
    y_fit, y_final_val = y_array[:-n_val], y_array[-n_val:]
    best_params["n_estimators"] = max_estimators
    best_params["early_stopping_rounds"] = early_stopping_rounds
    best_model = xgb.XGBRegressor(**best_params).fit(
        X_fit, y_fit, eval_set=[(X_final_val, y_final_val)], verbose=False
    )
    return best_model, study, best_params, float(study.best_value)

#calculate score if using current month as prediction for the next
naive_pred = test_data["demand_volume"].to_numpy()
results.append({"Tier": "Tier 1: Baseline", "Model": "Naive Baseline", "CV_RMSE": np.nan, "Best_Params": None, **evaluate_predictions(y_test, naive_pred)})
test_predictions["Tier 1: Baseline | Naive Baseline"] = naive_pred

#calculate score if using mean demand over training months as prediction for all test months
train_mean_pred = np.full(len(y_test), y_train.mean())
results.append({"Tier": "Tier 1: Baseline", "Model": "Training Mean Baseline", "CV_RMSE": np.nan, "Best_Params": None, **evaluate_predictions(y_test, train_mean_pred)})
test_predictions["Tier 1: Baseline | Training Mean Baseline"] = train_mean_pred

#loop to go over all models and model tiers, tunes them and scores
for tier_name, feats in feature_sets.items():
    print(f"\n{'='*100}\n{tier_name} (Features: {len(feats)})\n{'='*100}")
    X_tr, X_te = train_data[feats].copy(), test_data[feats].copy()

    print("\n[1/4] Tuning Elastic Net...")
    enet_search = tune_elastic_net(X_tr, y_train)
    p_enet = np.clip(enet_search.best_estimator_.predict(X_te), 0, None)
    model_name = "Elastic Net (Time-CV Tuned)"
    results.append({"Tier": tier_name, "Model": model_name, "CV_RMSE": float(-enet_search.best_score_), "Best_Params": enet_search.best_params_, **evaluate_predictions(y_test, p_enet)})
    fitted_models[f"{tier_name} | {model_name}"] = enet_search.best_estimator_
    test_predictions[f"{tier_name} | {model_name}"] = p_enet
    tuning_objects[f"{tier_name} | {model_name}"] = enet_search
    print(f"Best params: {enet_search.best_params_} \nCV RMSE: {float(-enet_search.best_score_):.4f}")

    print("\n[2/4] Tuning Poisson GLM...")
    pois_search = tune_poisson(X_tr, y_train)
    p_pois = np.clip(pois_search.best_estimator_.predict(X_te), 0, None)
    model_name = "Poisson GLM (Time-CV Tuned)"
    results.append({"Tier": tier_name, "Model": model_name, "CV_RMSE": float(-pois_search.best_score_), "Best_Params": pois_search.best_params_, **evaluate_predictions(y_test, p_pois)})
    fitted_models[f"{tier_name} | {model_name}"] = pois_search.best_estimator_
    test_predictions[f"{tier_name} | {model_name}"] = p_pois
    tuning_objects[f"{tier_name} | {model_name}"] = pois_search
    print(f"Best params: {pois_search.best_params_} \nCV RMSE: {float(-pois_search.best_score_):.4f}")

    print("\n[3/4] Tuning Random Forest with Optuna TPE...")
    rf_model, rf_study, rf_params, rf_cv_rmse = tune_rf_optuna(X_tr, y_train)
    p_rf = np.clip(rf_model.predict(X_te), 0, None)
    model_name = "Random Forest (Time-CV TPE Tuned)"
    results.append({"Tier": tier_name, "Model": model_name, "CV_RMSE": rf_cv_rmse, "Best_Params": rf_params, **evaluate_predictions(y_test, p_rf)})
    fitted_models[f"{tier_name} | {model_name}"] = rf_model
    test_predictions[f"{tier_name} | {model_name}"] = p_rf
    tuning_objects[f"{tier_name} | {model_name}"] = rf_study
    print(f"Best params: {rf_params} \nCV RMSE: {rf_cv_rmse:.4f}")

    print("\n[4/4] Tuning XGBoost with Optuna TPE...")
    xgb_model, xgb_study, xgb_params, xgb_cv_rmse = tune_xgb_optuna(X_tr, y_train)
    p_xgb = np.clip(xgb_model.predict(X_te), 0, None)
    model_name = "XGBoost (Time-CV TPE Tuned)"
    results.append({"Tier": tier_name, "Model": model_name, "CV_RMSE": xgb_cv_rmse, "Best_Params": xgb_params, **evaluate_predictions(y_test, p_xgb)})
    fitted_models[f"{tier_name} | {model_name}"] = xgb_model
    test_predictions[f"{tier_name} | {model_name}"] = p_xgb
    tuning_objects[f"{tier_name} | {model_name}"] = xgb_study
    print(f"Best params: {xgb_params} \nCV RMSE: {xgb_cv_rmse:.4f}")

leaderboard_df = pd.DataFrame(results).sort_values(["RMSE", "MAE"]).reset_index(drop=True)
print(f"\n{'='*110}\nFINAL HOLDOUT LEADERBOARD\n{'='*110}")
display(leaderboard_df[["Tier", "Model", "CV_RMSE", "RMSE", "MAE", "R2", "Best_Params"]].round(4))

### 15. Statistical Inference (Bootstrap 95% Confidence Intervals)
Perform bootstrap resampling on the holdout error distributions. This can be used to mathematically determines if the forecasting improvements are actually statistically significant

In [ ]:
def bootstrap_rmse_ci(y_true, y_pred, n_boot=500, random_state=42):
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)

    boot = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot[i] = rmse(y_true[idx], y_pred[idx])

    return {
        "RMSE": rmse(y_true, y_pred),
        "RMSE_CI_low": float(np.percentile(boot, 2.5)),
        "RMSE_CI_high": float(np.percentile(boot, 97.5)),
    }

ci_rows = []

for key, pred in test_predictions.items():
    tier_name, model_name = key.split(" | ", 1)
    out = bootstrap_rmse_ci(
        y_test.to_numpy(),
        pred,
        n_boot=500,
        random_state=RANDOM_STATE,
    )
    ci_rows.append({
        "Tier": tier_name,
        "Model": model_name,
        **out,
    })

rmse_ci_df = (
    pd.DataFrame(ci_rows)
      .sort_values("RMSE")
      .reset_index(drop=True)
)

print("=" * 90)
print("HOLDOUT RMSE WITH BOOTSTRAP 95% CI")
print("=" * 90)
display(rmse_ci_df.round(4))

### 16. Feature Importance
Extract the weights of the best tree model which can be interpreted as feature imporatnce.

In [ ]:
tree_rows = leaderboard_df[
    leaderboard_df["Model"].str.contains("Random Forest|XGBoost")
].copy()

if len(tree_rows):
    best_tree = tree_rows.iloc[0]
    best_key = f"{best_tree['Tier']} | {best_tree['Model']}"
    best_model = fitted_models[best_key]
    best_features = feature_sets[best_tree["Tier"]]

    if hasattr(best_model, "feature_importances_"):
        importance_df = (
            pd.DataFrame({
                "Feature": best_features,
                "Importance": best_model.feature_importances_,
            })
            .sort_values("Importance", ascending=False)
            .reset_index(drop=True)
        )

        print(f"Best tree model: {best_key}")
        display(importance_df.head(25).round(5))

In [ ]:
tree_rows = leaderboard_df[
    leaderboard_df["Model"].str.contains("Random Forest|XGBoost")
].copy()

if len(tree_rows):
    best_tree = tree_rows.iloc[1]
    best_key = f"{best_tree['Tier']} | {best_tree['Model']}"
    best_model = fitted_models[best_key]
    best_features = feature_sets[best_tree["Tier"]]

    if hasattr(best_model, "feature_importances_"):
        importance_df = (
            pd.DataFrame({
                "Feature": best_features,
                "Importance": best_model.feature_importances_,
            })
            .sort_values("Importance", ascending=False)
            .reset_index(drop=True)
        )

        print(f"Best tree model: {best_key}")
        display(importance_df.head(25).round(5))

### 17. Export Final Datasets
Export final data and results

In [ ]:
DF_OUTPUT_PATH = "/content/drive/MyDrive/Thesis Code/results_output/final_data_df.parquet"
MODEL_OUTPUT_PATH = "/content/drive/MyDrive/Thesis Code/results_output/final_data_model.parquet"
LEADERBOARD_OUTPUT_PATH = "/content/drive/MyDrive/Thesis Code/results_output/demand_forecast_leaderboard_V4.parquet"
CI_OUTPUT_PATH = "/content/drive/MyDrive/Thesis Code/results_output/demand_forecast_rmse_ci.parquet"

df.to_parquet(DF_OUTPUT_PATH, index=False)
model_complete.to_parquet(MODEL_OUTPUT_PATH, index=False)
leaderboard_df.to_parquet(LEADERBOARD_OUTPUT_PATH, index=False)
rmse_ci_df.to_parquet(CI_OUTPUT_PATH, index=False)